# EDA - Älvsjö och Hägersten 2021-2025

In [2]:
import pandas as pd
import duckdb
import matplotlib

df_raw = pd.read_excel('../data_files/brå_stockholm_21_25_raw.xls', header=None)

df_raw.head(10)


,0,1,2,3,4,5
0,Anmälda brott,NaN,Brottsförebyggande rådet,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,År,År,År,År,År
3,NaN,2021,2022,2023,2024,2025
4,NaN,/100 000 inv,/100 000 inv,/100 000 inv,/100 000 inv,/100 000 inv
5,Stockholm kommun,NaN,NaN,NaN,NaN,NaN
6,3-7 kap. Brott mot person,NaN,NaN,NaN,NaN,NaN
7,3 kap. Brott mot liv och hälsa,NaN,NaN,NaN,NaN,NaN
8,"Misshandel inkl. grov (5, 6 §)",1004,978,1017,1039,1072
9,3-7 kap. Brott mot person,NaN,NaN,NaN,NaN,NaN


In [4]:
# Döp om  kolumn '0' till 'Brottstyp'

df_cleaned = df_raw.rename(columns={0: 'Brottstyp'})

df_cleaned.head(10)

,Brottstyp,1,2,3,4,5
0,Anmälda brott,NaN,Brottsförebyggande rådet,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,År,År,År,År,År
3,NaN,2021,2022,2023,2024,2025
4,NaN,/100 000 inv,/100 000 inv,/100 000 inv,/100 000 inv,/100 000 inv
5,Stockholm kommun,NaN,NaN,NaN,NaN,NaN
6,3-7 kap. Brott mot person,NaN,NaN,NaN,NaN,NaN
7,3 kap. Brott mot liv och hälsa,NaN,NaN,NaN,NaN,NaN
8,"Misshandel inkl. grov (5, 6 §)",1004,978,1017,1039,1072
9,3-7 kap. Brott mot person,NaN,NaN,NaN,NaN,NaN


In [6]:
# Lägg till årtal i kolumn 1-5

years = [df_cleaned.iloc[3, i] for i in range(1, len(df_cleaned.columns))]

df_cleaned.columns = ['Brottstyp'] + years

df_cleaned.head(10)

,Brottstyp,2021,2022,2023,2024,2025
0,Anmälda brott,NaN,Brottsförebyggande rådet,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,År,År,År,År,År
3,NaN,2021,2022,2023,2024,2025
4,NaN,/100 000 inv,/100 000 inv,/100 000 inv,/100 000 inv,/100 000 inv
5,Stockholm kommun,NaN,NaN,NaN,NaN,NaN
6,3-7 kap. Brott mot person,NaN,NaN,NaN,NaN,NaN
7,3 kap. Brott mot liv och hälsa,NaN,NaN,NaN,NaN,NaN
8,"Misshandel inkl. grov (5, 6 §)",1004,978,1017,1039,1072
9,3-7 kap. Brott mot person,NaN,NaN,NaN,NaN,NaN


In [ ]:
print(df_cleaned.dtypes)

Brottstyp       str
2021         object
2022         object
2023         object
2024         object
2025         object
dtype: object


In [8]:
# Justera data type för brottsdatan per capita (str -> int) 

year_cols = ['2021', '2022', '2023', '2024', '2025']
df_cleaned[year_cols] = df_cleaned[year_cols].apply(pd.to_numeric, errors='coerce')

In [9]:
print(df_cleaned.dtypes)

Brottstyp        str
2021         float64
2022         float64
2023         float64
2024         float64
2025         float64
dtype: object


In [18]:
list(df_cleaned[df_cleaned[['2021','2022','2023','2024','2025']].notna().any(axis=1)]['Brottstyp'])


[nan,
 'Misshandel inkl. grov (5, 6 §)',
 '4 kap. Brott mot frihet och frid',
 '6 kap. Sexualbrott',
 'Tillgrepp av motordrivet fortskaffningsmedel (7 §)',
 'Cykel',
 'Inbrottsstöld i bostad m.m.',
 'Från fordon m.m.',
 'Rån, inkl. grovt (5, 6 §)',
 '12 kap. Skadegörelsebrott',
 'Brott mot narkotikastrafflagen']

In [21]:
crime_types = [
    'Misshandel inkl. grov (5, 6 §)',
    '4 kap. Brott mot frihet och frid',
    '6 kap. Sexualbrott',
    'Tillgrepp av motordrivet fortskaffningsmedel (7 §)',
    'Cykel',
    'Från fordon m.m.',
    'Inbrottsstöld i bostad m.m.',
    'Rån, inkl. grovt (5, 6 §)',
    '12 kap. Skadegörelsebrott',
    'Brott mot narkotikastrafflagen'
]

In [23]:
df_filtered = df_cleaned[df_cleaned['Brottstyp'].isin(crime_types)]

In [25]:
df_cleaned = duckdb.sql("""--sql
SELECT
    CASE Brottstyp
        WHEN 'Misshandel inkl. grov (5, 6 §)' THEN 'Misshandel'
        WHEN '4 kap. Brott mot frihet och frid' THEN 'Hot & olaga intrång'
        WHEN '6 kap. Sexualbrott' THEN 'Sexualbrott'
        WHEN 'Tillgrepp av motordrivet fortskaffningsmedel (7 §)' THEN 'Bilstöld'
        WHEN 'Cykel' THEN 'Cykelstöld'
        WHEN 'Från fordon m.m.' THEN 'Stöld från fordon'
        WHEN 'Inbrottsstöld i bostad m.m.' THEN 'Bostadsinbrott'
        WHEN 'Rån, inkl. grovt (5, 6 §)' THEN 'Rån'
        WHEN '12 kap. Skadegörelsebrott' THEN 'Skadegörelse'
        WHEN 'Brott mot narkotikastrafflagen' THEN 'Narkotikabrott'
    END AS Brottstyp,
        * EXCLUDE (Brottstyp)              
FROM df_filtered
""").df()

In [26]:
df_cleaned.head(10)

,Brottstyp,2021,2022,2023,2024,2025
0,Misshandel,1004.0,978.0,1017.0,1039.0,1072.0
1,Hot & olaga intrång,1380.0,1309.0,1511.0,1756.0,1792.0
2,Sexualbrott,257.0,240.0,240.0,257.0,283.0
3,Bilstöld,191.0,171.0,138.0,122.0,85.0
4,Cykelstöld,940.0,699.0,645.0,653.0,580.0
5,Bostadsinbrott,485.0,399.0,376.0,397.0,360.0
6,Stöld från fordon,808.0,652.0,645.0,432.0,362.0
7,Rån,132.0,104.0,97.0,66.0,50.0
8,Skadegörelse,8819.0,7637.0,8939.0,7379.0,6208.0
9,Narkotikabrott,1462.0,1547.0,1537.0,1655.0,1642.0


In [27]:
df_cleaned.to_csv('../data_files/brottsstatistik_stockholm_cleaned.csv', index=False)

# EDA - alla kommuner 2025

In [30]:
df_sthlm_raw = pd.read_excel('../data_files/brå_alla_kommuner_2025_raw.xls', header=None)

df_sthlm_raw.head(20)

,0,1,2
0,Anmälda brott,NaN,Brottsförebyggande rådet
1,NaN,NaN,NaN
2,NaN,År,NaN
3,NaN,2025,NaN
4,NaN,/100 000 inv,NaN
5,Stadsdelsområde Bromma (Sthlm),NaN,NaN
6,3-7 kap. Brott mot person,NaN,NaN
7,3 kap. Brott mot liv och hälsa,NaN,NaN
8,"Misshandel inkl. grov (5, 6 §)",603,NaN
9,3-7 kap. Brott mot person,NaN,NaN


In [36]:
# Döp om  kolumn '0' till 'Brottstyp'

df_sthlm_columns = df_sthlm_raw.rename(columns={0: 'Brottstyp'})

df_sthlm_columns.head(10)

,Brottstyp,1,2
0,Anmälda brott,NaN,Brottsförebyggande rådet
1,NaN,NaN,NaN
2,NaN,År,NaN
3,NaN,2025,NaN
4,NaN,/100 000 inv,NaN
5,Stadsdelsområde Bromma (Sthlm),NaN,NaN
6,3-7 kap. Brott mot person,NaN,NaN
7,3 kap. Brott mot liv och hälsa,NaN,NaN
8,"Misshandel inkl. grov (5, 6 §)",603,NaN
9,3-7 kap. Brott mot person,NaN,NaN


In [37]:
df_sthlm_columns = df_sthlm_columns.rename(columns={1: '/100 000 inv'})

df_sthlm_columns.head(10)

,Brottstyp,/100 000 inv,2
0,Anmälda brott,NaN,Brottsförebyggande rådet
1,NaN,NaN,NaN
2,NaN,År,NaN
3,NaN,2025,NaN
4,NaN,/100 000 inv,NaN
5,Stadsdelsområde Bromma (Sthlm),NaN,NaN
6,3-7 kap. Brott mot person,NaN,NaN
7,3 kap. Brott mot liv och hälsa,NaN,NaN
8,"Misshandel inkl. grov (5, 6 §)",603,NaN
9,3-7 kap. Brott mot person,NaN,NaN


In [55]:
# För att vi ska kunna filtrera på område

df_sthlm_columns['radnr'] = range(len(df_sthlm_columns))

df_sthlm_columns.head(100)

,Brottstyp,/100 000 inv,2,radnr
0,Anmälda brott,NaN,Brottsförebyggande rådet,0
1,NaN,NaN,NaN,1
2,NaN,År,NaN,2
3,NaN,2025,NaN,3
4,NaN,/100 000 inv,NaN,4
...,...,...,...,...
95,3-7 kap. Brott mot person,NaN,NaN,95
96,6 kap. Sexualbrott,279,NaN,96
97,8-12 kap. Brott mot förmögenhet,NaN,NaN,97
98,"8 kap. Stöld, rån m.m.",NaN,NaN,98


In [56]:
# Hitta vilka rader som gälller för resp. kommun

df_sthlm_columns[df_sthlm_columns['Brottstyp'].str.contains('Bromma|Enskede|Farsta|Hägersten|Hässelby|Järva|Kungsholmen|Norra|Skarpnäck|Skärholmen|Södermalm')]

,Brottstyp,/100 000 inv,2,radnr
5,Stadsdelsområde Bromma (Sthlm),NaN,NaN,5
33,Stadsdelsområde Enskede - Årsta - Vantör (Sthlm),NaN,NaN,33
61,Stadsdelsområde Farsta (Sthlm),NaN,NaN,61
89,Stadsdelsområde Hägersten-Älvsjö (Sthlm),NaN,NaN,89
117,Stadsdelsområde Hässelby - Vällingby (Sthlm),NaN,NaN,117
145,Stadsdelsområde Järva (Sthlm),NaN,NaN,145
173,Stadsdelsområde Kungsholmen (Sthlm),NaN,NaN,173
201,Stadsdelsområde Norra innerstaden (Sthlm),NaN,NaN,201
229,Stadsdelsområde Skarpnäck (Sthlm),NaN,NaN,229
257,Stadsdelsområde Skärholmen (Sthlm),NaN,NaN,257


In [57]:
BROMMA_START = 5
ENSKEDE_START = 33
FARSTA_START = 61
ÄLVJSÖ_START = 89
HÄSSELBY_START = 117
JÄRVA_START = 145
KUNGSHOLMEN_START = 173
NORRA_INNERSTADEN_START = 201
SKARPNÄCK_START = 229
SKÄRHOLMEN_START = 257
SÖDERMALM_START = 285

df_cleaned = duckdb.sql(f"""--sql
SELECT 
    *,
    CASE 
        WHEN radnr >= {BROMMA_START}             AND radnr < {ENSKEDE_START}           THEN 'Bromma'
        WHEN radnr >= {ENSKEDE_START}            AND radnr < {FARSTA_START}            THEN 'Enskede - Årsta - Vantör'
        WHEN radnr >= {FARSTA_START}             AND radnr < {ÄLVJSÖ_START}            THEN 'Farsta' 
        WHEN radnr >= {ÄLVJSÖ_START}             AND radnr < {HÄSSELBY_START}          THEN 'Hägersten - Älvsjö'
        WHEN radnr >= {HÄSSELBY_START}           AND radnr < {JÄRVA_START}             THEN 'Hässelby - Vällingby'
        WHEN radnr >= {JÄRVA_START}              AND radnr < {KUNGSHOLMEN_START}       THEN 'Järva'
        WHEN radnr >= {KUNGSHOLMEN_START}        AND radnr < {NORRA_INNERSTADEN_START} THEN 'Kungsholmen'
        WHEN radnr >= {NORRA_INNERSTADEN_START}  AND radnr < {SKARPNÄCK_START}         THEN 'Norra innerstaden'
        WHEN radnr >= {SKARPNÄCK_START}          AND radnr < {SKÄRHOLMEN_START}        THEN 'Skarpnäck'
        WHEN radnr >= {SKÄRHOLMEN_START}         AND radnr < {SÖDERMALM_START}         THEN 'Skärholmen'
        WHEN radnr >= {SÖDERMALM_START}                                                THEN 'Södermalm'
    END AS Stadsdelsområde
FROM df_sthlm_columns
""").df()

df_cleaned.head(10)

,Brottstyp,/100 000 inv,2,radnr,Stadsdelsområde
0,Anmälda brott,NaN,Brottsförebyggande rådet,0,NaN
1,NaN,NaN,NaN,1,NaN
2,NaN,År,NaN,2,NaN
3,NaN,2025,NaN,3,NaN
4,NaN,/100 000 inv,NaN,4,NaN
5,Stadsdelsområde Bromma (Sthlm),NaN,NaN,5,Bromma
6,3-7 kap. Brott mot person,NaN,NaN,6,Bromma
7,3 kap. Brott mot liv och hälsa,NaN,NaN,7,Bromma
8,"Misshandel inkl. grov (5, 6 §)",603,NaN,8,Bromma
9,3-7 kap. Brott mot person,NaN,NaN,9,Bromma


In [58]:
df_cleaned = df_cleaned.dropna(subset=['/100 000 inv'])

In [59]:
df_cleaned.head(10)

,Brottstyp,/100 000 inv,2,radnr,Stadsdelsområde
2,NaN,År,NaN,2,NaN
3,NaN,2025,NaN,3,NaN
4,NaN,/100 000 inv,NaN,4,NaN
8,"Misshandel inkl. grov (5, 6 §)",603,NaN,8,Bromma
10,4 kap. Brott mot frihet och frid,1372,NaN,10,Bromma
12,6 kap. Sexualbrott,253,NaN,12,Bromma
15,Tillgrepp av motordrivet fortskaffningsmedel (...,102,NaN,15,Bromma
19,Cykel,600,NaN,19,Bromma
22,"Stöld genom inbrott, inbrottsstöld i bostad, e...",646,NaN,22,Bromma
26,Från fordon m.m.,444,NaN,26,Bromma


In [60]:
crime_types = [
    'Misshandel inkl. grov (5, 6 §)',
    '4 kap. Brott mot frihet och frid',
    '6 kap. Sexualbrott',
    'Tillgrepp av motordrivet fortskaffningsmedel (7 §)',
    'Cykel',
    'Från fordon m.m.',
    'Inbrottsstöld i bostad m.m.',
    'Rån, inkl. grovt (5, 6 §)',
    '12 kap. Skadegörelsebrott',
    'Brott mot narkotikastrafflagen'
]

In [61]:
df_filtered = df_cleaned[df_cleaned['Brottstyp'].isin(crime_types)]

In [62]:
df_cleaned = duckdb.sql("""--sql
SELECT
    CASE Brottstyp
        WHEN 'Misshandel inkl. grov (5, 6 §)' THEN 'Misshandel'
        WHEN '4 kap. Brott mot frihet och frid' THEN 'Hot & olaga intrång'
        WHEN '6 kap. Sexualbrott' THEN 'Sexualbrott'
        WHEN 'Tillgrepp av motordrivet fortskaffningsmedel (7 §)' THEN 'Bilstöld'
        WHEN 'Cykel' THEN 'Cykelstöld'
        WHEN 'Från fordon m.m.' THEN 'Stöld från fordon'
        WHEN 'Inbrottsstöld i bostad m.m.' THEN 'Bostadsinbrott'
        WHEN 'Rån, inkl. grovt (5, 6 §)' THEN 'Rån'
        WHEN '12 kap. Skadegörelsebrott' THEN 'Skadegörelse'
        WHEN 'Brott mot narkotikastrafflagen' THEN 'Narkotikabrott'
    END AS Brottstyp,
        * EXCLUDE (Brottstyp)              
FROM df_filtered
""").df()

In [63]:
df_cleaned.head(50)

,Brottstyp,/100 000 inv,2,radnr,Stadsdelsområde
0,Misshandel,603,None,8,Bromma
1,Hot & olaga intrång,1372,None,10,Bromma
2,Sexualbrott,253,None,12,Bromma
3,Bilstöld,102,None,15,Bromma
4,Cykelstöld,600,None,19,Bromma
5,Stöld från fordon,444,None,26,Bromma
6,Rån,23,None,29,Bromma
7,Skadegörelse,3957,None,31,Bromma
8,Narkotikabrott,735,None,32,Bromma
9,Misshandel,1033,None,36,Enskede - Årsta - Vantör


In [64]:
df_cleaned = df_cleaned.drop(columns=['2', 'radnr'])

In [68]:
df_cleaned.head(40)

,Brottstyp,/100 000 inv,Stadsdelsområde
0,Misshandel,603,Bromma
1,Hot & olaga intrång,1372,Bromma
2,Sexualbrott,253,Bromma
3,Bilstöld,102,Bromma
4,Cykelstöld,600,Bromma
5,Stöld från fordon,444,Bromma
6,Rån,23,Bromma
7,Skadegörelse,3957,Bromma
8,Narkotikabrott,735,Bromma
9,Misshandel,1033,Enskede - Årsta - Vantör


In [70]:
df_cleaned.to_csv('../data_files/brå_alla_kommuner_2025_cleaned.csv', index=False)

# EDA - Alla kommuner 2021-2015

In [8]:
df_all_raw = pd.read_excel('../data_files/bra_alla_kommuner_2021_2025.xls', header=None)

df_all_raw.head(10)

,0,1,2,3,4,5
0,Anmälda brott,NaN,Brottsförebyggande rådet,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,År,År,År,År,År
3,NaN,2021,2022,2023,2024,2025
4,NaN,/100 000 inv,/100 000 inv,/100 000 inv,/100 000 inv,/100 000 inv
5,Stockholm kommun,NaN,NaN,NaN,NaN,NaN
6,Totalt antal brott,21308,19601,21402,20419,19159
7,3-7 kap. Brott mot person,NaN,NaN,NaN,NaN,NaN
8,3 kap. Brott mot liv och hälsa,NaN,NaN,NaN,NaN,NaN
9,"Försök till mord eller dråp (1, 2 §)",13,13,17,16,14


In [ ]:
# Lägg till kolumnnamn

df_all_columns = df_all_raw.rename(columns={0: 'Brottstyp', 1:'2021', 2:'2022', 3:'2023', 4:'2024', 5:'2025'})

In [10]:
df_all_columns.head(20)

,Brottstyp,2021,2022,2023,2024,2025
0,Anmälda brott,NaN,Brottsförebyggande rådet,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,År,År,År,År,År
3,NaN,2021,2022,2023,2024,2025
4,NaN,/100 000 inv,/100 000 inv,/100 000 inv,/100 000 inv,/100 000 inv
5,Stockholm kommun,NaN,NaN,NaN,NaN,NaN
6,Totalt antal brott,21308,19601,21402,20419,19159
7,3-7 kap. Brott mot person,NaN,NaN,NaN,NaN,NaN
8,3 kap. Brott mot liv och hälsa,NaN,NaN,NaN,NaN,NaN
9,"Försök till mord eller dråp (1, 2 §)",13,13,17,16,14


In [12]:
# Lägg till radnr

df_all_columns['radnr'] = range(len(df_all_columns))

df_all_columns.head(20)

,Brottstyp,2021,2022,2023,2024,2025,radnr
0,Anmälda brott,NaN,Brottsförebyggande rådet,NaN,NaN,NaN,0
1,NaN,NaN,NaN,NaN,NaN,NaN,1
2,NaN,År,År,År,År,År,2
3,NaN,2021,2022,2023,2024,2025,3
4,NaN,/100 000 inv,/100 000 inv,/100 000 inv,/100 000 inv,/100 000 inv,4
5,Stockholm kommun,NaN,NaN,NaN,NaN,NaN,5
6,Totalt antal brott,21308,19601,21402,20419,19159,6
7,3-7 kap. Brott mot person,NaN,NaN,NaN,NaN,NaN,7
8,3 kap. Brott mot liv och hälsa,NaN,NaN,NaN,NaN,NaN,8
9,"Försök till mord eller dråp (1, 2 §)",13,13,17,16,14,9


In [ ]:
# Hitta vilka rader som gälller för resp. område

df_all_columns[df_all_columns['Brottstyp'].str.contains('Stockholm|Bromma|Enskede|Farsta|Hägersten|Hässelby|Järva|Kungsholmen|Norra|Skarpnäck|Skärholmen|Södermalm')]

,Brottstyp,2021,2022,2023,2024,2025,radnr
5,Stockholm kommun,NaN,NaN,NaN,NaN,NaN,5
69,Stadsdelsområde Bromma (Sthlm),NaN,NaN,NaN,NaN,NaN,69
133,Stadsdelsområde Enskede - Årsta - Vantör (Sthlm),NaN,NaN,NaN,NaN,NaN,133
197,Stadsdelsområde Farsta (Sthlm),NaN,NaN,NaN,NaN,NaN,197
261,Stadsdelsområde Hägersten-Älvsjö (Sthlm),NaN,NaN,NaN,NaN,NaN,261
325,Stadsdelsområde Hässelby - Vällingby (Sthlm),NaN,NaN,NaN,NaN,NaN,325
389,Stadsdelsområde Järva (Sthlm),NaN,NaN,NaN,NaN,NaN,389
453,Stadsdelsområde Kungsholmen (Sthlm),NaN,NaN,NaN,NaN,NaN,453
517,Stadsdelsområde Norra innerstaden (Sthlm),NaN,NaN,NaN,NaN,NaN,517
581,Stadsdelsområde Skarpnäck (Sthlm),NaN,NaN,NaN,NaN,NaN,581


In [ ]:
# Lägg till kolumn för stadsdelsområde

STHLM_TOTAL_START = 7
BROMMA_START = 69
ENSKEDE_START = 133
FARSTA_START = 197
ÄLVJSÖ_START = 261
HÄSSELBY_START = 325
JÄRVA_START = 389
KUNGSHOLMEN_START = 453
NORRA_INNERSTADEN_START = 517
SKARPNÄCK_START = 581
SKÄRHOLMEN_START = 645
SÖDERMALM_START = 709

df_all_areas = duckdb.sql(f"""--sql
SELECT 
    *,
    CASE 
        WHEN radnr >= {STHLM_TOTAL_START}        AND radnr < {BROMMA_START}            THEN 'Stockholms kommun total'       
        WHEN radnr >= {BROMMA_START}             AND radnr < {ENSKEDE_START}           THEN 'Bromma'
        WHEN radnr >= {ENSKEDE_START}            AND radnr < {FARSTA_START}            THEN 'Enskede - Årsta - Vantör'
        WHEN radnr >= {FARSTA_START}             AND radnr < {ÄLVJSÖ_START}            THEN 'Farsta' 
        WHEN radnr >= {ÄLVJSÖ_START}             AND radnr < {HÄSSELBY_START}          THEN 'Hägersten - Älvsjö'
        WHEN radnr >= {HÄSSELBY_START}           AND radnr < {JÄRVA_START}             THEN 'Hässelby - Vällingby'
        WHEN radnr >= {JÄRVA_START}              AND radnr < {KUNGSHOLMEN_START}       THEN 'Järva'
        WHEN radnr >= {KUNGSHOLMEN_START}        AND radnr < {NORRA_INNERSTADEN_START} THEN 'Kungsholmen'
        WHEN radnr >= {NORRA_INNERSTADEN_START}  AND radnr < {SKARPNÄCK_START}         THEN 'Norra innerstaden'
        WHEN radnr >= {SKARPNÄCK_START}          AND radnr < {SKÄRHOLMEN_START}        THEN 'Skarpnäck'
        WHEN radnr >= {SKÄRHOLMEN_START}         AND radnr < {SÖDERMALM_START}         THEN 'Skärholmen'
        WHEN radnr >= {SÖDERMALM_START}                                                THEN 'Södermalm'
    END AS Stadsdelsområde
FROM df_all_columns
""").df()

df_all_areas.head(10)

,Brottstyp,2021,2022,2023,2024,2025,radnr,Stadsdelsområde
0,Anmälda brott,NaN,Brottsförebyggande rådet,NaN,NaN,NaN,0,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,1,NaN
2,NaN,År,År,År,År,År,2,NaN
3,NaN,2021,2022,2023,2024,2025,3,NaN
4,NaN,/100 000 inv,/100 000 inv,/100 000 inv,/100 000 inv,/100 000 inv,4,NaN
5,Stockholm kommun,NaN,NaN,NaN,NaN,NaN,5,Stockholms kommun total
6,Totalt antal brott,21308,19601,21402,20419,19159,6,Stockholms kommun total
7,3-7 kap. Brott mot person,NaN,NaN,NaN,NaN,NaN,7,Stockholms kommun total
8,3 kap. Brott mot liv och hälsa,NaN,NaN,NaN,NaN,NaN,8,Stockholms kommun total
9,"Försök till mord eller dråp (1, 2 §)",13,13,17,16,14,9,Stockholms kommun total


In [36]:
# Ta bort toma rader

df_all_areas = df_all_areas.dropna(subset=['2021', '2022', '2023', '2024', '2025'])
df_all_areas = df_all_areas[df_all_areas['Brottstyp'].notna()]

df_all_areas.head(20)

,Brottstyp,2021,2022,2023,2024,2025,radnr,Stadsdelsområde
6,Totalt antal brott,21308,19601,21402,20419,19159,6,Stockholms kommun total
9,"Försök till mord eller dråp (1, 2 §)",13,13,17,16,14,9,Stockholms kommun total
12,"Misshandel inkl. grov (5, 6 §)",1004,978,1017,1039,1072,12,Stockholms kommun total
16,Människorov (1 §),7,5,5,6,4,16,Stockholms kommun total
20,Olaga frihetsberövande (2§) (fr o m 2020),19,15,16,18,18,20,Stockholms kommun total
23,Olaga tvång (4 §),11,9,10,9,10,23,Stockholms kommun total
26,Grov fridskränkning (4 a §),19,17,21,16,12,26,Stockholms kommun total
29,Olaga förföljelse (4 b §),5,4,6,7,10,29,Stockholms kommun total
32,Olaga hot (5 §),450,414,438,450,452,32,Stockholms kommun total
35,"Hemfridsbrott, olaga intrång (6 §)",127,113,111,125,142,35,Stockholms kommun total


In [35]:
# Kolla vad det är för data types

df_all_areas.dtypes

Brottstyp            str
2021                 str
2022                 str
2023                 str
2024                 str
2025                 str
radnr              int64
Stadsdelsområde      str
dtype: object

In [37]:
# Ändra från str till int

cols = ['2021', '2022', '2023', '2024', '2025']
df_all_areas[cols] = df_all_areas[cols].apply(pd.to_numeric, errors='coerce').astype(int)

In [ ]:
# Lista unika brottstyper 

df_all_areas['Brottstyp'].unique()

<ArrowStringArray>
[                                                           'Totalt antal brott',
                                          'Försök till mord eller dråp (1, 2 §)',
                                                'Misshandel inkl. grov (5, 6 §)',
                                                             'Människorov (1 §)',
                                     'Olaga frihetsberövande (2§) (fr o m 2020)',
                                                             'Olaga tvång (4 §)',
                                                   'Grov fridskränkning (4 a §)',
                                                     'Olaga förföljelse (4 b §)',
                                                               'Olaga hot (5 §)',
                                            'Hemfridsbrott, olaga intrång (6 §)',
                                         'Olaga integritetsintrång (6 c, 6 d §)',
                                                               'Ofredande (7 §)

In [ ]:
# Ändra namn på brotstyper

df_all_rename = duckdb.sql("""--sql
SELECT
    CASE Brottstyp
        WHEN 'Försök till mord eller dråp (1, 2 §)' THEN 'Mordförsök'
        WHEN 'Misshandel inkl. grov (5, 6 §)' THEN 'Misshandel'
        WHEN 'Rån, inkl. grovt (5, 6 §)' THEN 'Rån'
        WHEN 'Olaga hot (5 §)' THEN 'Hot & ofredande'
        WHEN 'Ofredande (7 §)' THEN 'Hot & ofredande'
        WHEN 'Människorov (1 §)' THEN 'Tvång & frihetsbrott'
        WHEN 'Olaga frihetsberövande (2§) (fr o m 2020)' THEN 'Tvång & frihetsbrott'
        WHEN 'Olaga tvång (4 §)' THEN 'Tvång & frihetsbrott'
        WHEN 'Grov fridskränkning (4 a §)' THEN 'Tvång & frihetsbrott'
        WHEN 'Olaga förföljelse (4 b §)' THEN 'Tvång & frihetsbrott'
        WHEN 'Hemfridsbrott, olaga intrång (6 §)' THEN 'Intrång & integritet'
        WHEN 'Olaga integritetsintrång (6 c, 6 d §)' THEN 'Intrång & integritet'
        WHEN '6 kap. Sexualbrott' THEN 'Sexualbrott'
        WHEN 'Tillgrepp av motordrivet fortskaffningsmedel (7 §)' THEN 'Bilstöld'
        WHEN 'Stöld av icke motordrivet fortskaffningsmedel (1, 2, 4 §)' THEN 'Cykelstöld'
        WHEN 'Stöld genom inbrott, inbrottsstöld i bostad, ej av skjutvapen (1, 2, 4, 4 a§)' THEN 'Bostadsinbrott'
        WHEN 'Genom brand' THEN 'Skadegörelse'
        WHEN 'På motorfordon (ej genom brand)' THEN 'Skadegörelse'
        WHEN 'Klotter' THEN 'Skadegörelse'
        WHEN 'Brott mot narkotikastrafflagen' THEN 'Narkotikabrott'
        WHEN 'Totalt antal brott' THEN 'Totalt antal brott'
    END AS Brottstyp,
        * EXCLUDE (Brottstyp)              
FROM df_all_areas
""").df()

In [43]:
df_all_rename.head(30)

,Brottstyp,2021,2022,2023,2024,2025,radnr,Stadsdelsområde
0,Totalt antal brott,21308,19601,21402,20419,19159,6,Stockholms kommun total
1,Mordförsök,13,13,17,16,14,9,Stockholms kommun total
2,Misshandel,1004,978,1017,1039,1072,12,Stockholms kommun total
3,Tvång & frihetsbrott,7,5,5,6,4,16,Stockholms kommun total
4,Tvång & frihetsbrott,19,15,16,18,18,20,Stockholms kommun total
5,Tvång & frihetsbrott,11,9,10,9,10,23,Stockholms kommun total
6,Tvång & frihetsbrott,19,17,21,16,12,26,Stockholms kommun total
7,Tvång & frihetsbrott,5,4,6,7,10,29,Stockholms kommun total
8,Hot & ofredande,450,414,438,450,452,32,Stockholms kommun total
9,Intrång & integritet,127,113,111,125,142,35,Stockholms kommun total


In [ ]:
# Slå ihop brotstyper med samma namn

df_all_grouped = duckdb.sql("""--sql
SELECT
    Brottstyp,
    Stadsdelsområde,
    SUM("2021") AS "2021",
    SUM("2022") AS "2022",
    SUM("2023") AS "2023",
    SUM("2024") AS "2024",
    SUM("2025") AS "2025"
FROM df_all_rename
GROUP BY Brottstyp, Stadsdelsområde
""").df()

In [ ]:
# Check att det ser korrekt ut

df_all_grouped[df_all_grouped['Brottstyp'] == 'Tvång & frihetsbrott']

,Brottstyp,Stadsdelsområde,2021,2022,2023,2024,2025
2,Tvång & frihetsbrott,Kungsholmen,40.0,14.0,23.0,37.0,29.0
25,Tvång & frihetsbrott,Bromma,47.0,33.0,58.0,34.0,27.0
39,Tvång & frihetsbrott,Hässelby - Vällingby,81.0,69.0,64.0,44.0,82.0
46,Tvång & frihetsbrott,Skärholmen,103.0,86.0,141.0,90.0,114.0
54,Tvång & frihetsbrott,Enskede - Årsta - Vantör,75.0,56.0,46.0,63.0,53.0
57,Tvång & frihetsbrott,Hägersten - Älvsjö,47.0,36.0,43.0,48.0,35.0
77,Tvång & frihetsbrott,Järva,0.0,0.0,0.0,109.0,96.0
80,Tvång & frihetsbrott,Skarpnäck,35.0,49.0,51.0,69.0,50.0
85,Tvång & frihetsbrott,Södermalm,55.0,39.0,43.0,41.0,48.0
107,Tvång & frihetsbrott,Farsta,66.0,46.0,88.0,44.0,41.0


In [53]:
df_all_grouped.head(40)

,Brottstyp,Stadsdelsområde,2021,2022,2023,2024,2025
0,Totalt antal brott,Enskede - Årsta - Vantör,22897.0,20004.0,22782.0,20725.0,19418.0
1,Totalt antal brott,Hägersten - Älvsjö,19828.0,17446.0,17099.0,15583.0,15377.0
2,Tvång & frihetsbrott,Kungsholmen,40.0,14.0,23.0,37.0,29.0
3,Sexualbrott,Kungsholmen,219.0,202.0,241.0,256.0,211.0
4,Rån,Kungsholmen,59.0,62.0,48.0,41.0,35.0
5,Bostadsinbrott,Farsta,946.0,797.0,626.0,700.0,668.0
6,Narkotikabrott,Farsta,985.0,1092.0,1247.0,1450.0,1097.0
7,Intrång & integritet,Hässelby - Vällingby,113.0,109.0,108.0,98.0,140.0
8,Hot & ofredande,Järva,0.0,0.0,0.0,1347.0,1361.0
9,Skadegörelse,Järva,0.0,0.0,0.0,2514.0,1435.0


In [55]:
df_all_cleaned = df_all_grouped.sort_values(['Stadsdelsområde', 'Brottstyp']).reset_index(drop=True)

df_all_cleaned.head(20)

,Brottstyp,Stadsdelsområde,2021,2022,2023,2024,2025
0,Bilstöld,Bromma,245.0,250.0,171.0,157.0,102.0
1,Bostadsinbrott,Bromma,876.0,774.0,664.0,639.0,646.0
2,Cykelstöld,Bromma,926.0,652.0,652.0,742.0,673.0
3,Hot & ofredande,Bromma,737.0,554.0,768.0,876.0,941.0
4,Intrång & integritet,Bromma,131.0,114.0,94.0,120.0,117.0
5,Misshandel,Bromma,581.0,558.0,615.0,617.0,603.0
6,Mordförsök,Bromma,7.0,2.0,12.0,7.0,16.0
7,Narkotikabrott,Bromma,599.0,690.0,622.0,662.0,735.0
8,Rån,Bromma,69.0,66.0,56.0,37.0,23.0
9,Sexualbrott,Bromma,209.0,211.0,230.0,244.0,253.0


In [57]:
df_all_cleaned.to_csv('../data_files/bra_alla_kommuner_2021-2025_cleaned.csv', index=False)